In [1]:
import nltk
import numpy as np
nltk.download('brown')
from nltk.corpus import brown

[nltk_data] Downloading package brown to /usr/share/nltk_data...
[nltk_data]   Package brown is already up-to-date!


In [2]:
sentances = brown.sents()
sentances

[['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', "Atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that', 'any', 'irregularities', 'took', 'place', '.'], ['The', 'jury', 'further', 'said', 'in', 'term-end', 'presentments', 'that', 'the', 'City', 'Executive', 'Committee', ',', 'which', 'had', 'over-all', 'charge', 'of', 'the', 'election', ',', '``', 'deserves', 'the', 'praise', 'and', 'thanks', 'of', 'the', 'City', 'of', 'Atlanta', "''", 'for', 'the', 'manner', 'in', 'which', 'the', 'election', 'was', 'conducted', '.'], ...]

In [3]:
# Text preprocessing
import re

def is_word(token):
    return re.search(r"[a-z]", token)

preprocessed_corpus = []
for sent in sentances:
    preprocessed_sentance = []
    for word in sent:
        word = word.lower()
        if(is_word(word)):
            preprocessed_sentance.append(word)

    preprocessed_corpus.append(preprocessed_sentance)
    
preprocessed_corpus[:1]

[['the',
  'fulton',
  'county',
  'grand',
  'jury',
  'said',
  'friday',
  'an',
  'investigation',
  'of',
  "atlanta's",
  'recent',
  'primary',
  'election',
  'produced',
  'no',
  'evidence',
  'that',
  'any',
  'irregularities',
  'took',
  'place']]

In [4]:
counts = {}
for sent in preprocessed_corpus:
    for word in sent:
        if (word not in counts):
            counts[word] = 1
        else:
            counts[word] += 1


In [5]:
min_freq = 2
vocab = set()
for word in counts:
    if counts[word] >= 2:
        vocab.add(word)

sorted(vocab)
print(len(vocab))

27131


In [6]:
filtered_sentances = []
for sent in preprocessed_corpus:
    filtered_sent = [w for w in sent if w in vocab]
    filtered_sentances.append(filtered_sent)

filtered_sentances[:1]

[['the',
  'fulton',
  'county',
  'grand',
  'jury',
  'said',
  'friday',
  'an',
  'investigation',
  'of',
  "atlanta's",
  'recent',
  'primary',
  'election',
  'produced',
  'no',
  'evidence',
  'that',
  'any',
  'irregularities',
  'took',
  'place']]

In [7]:
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

word2idx['the'], idx2word[20013]

(22504, 'oriented')

In [8]:
word_counts = []
for w in vocab:
    word_counts.append(counts[w])

word_counts = np.array(word_counts)
print(len(word_counts))

27131


In [9]:
word_counts[word2idx['the']]

np.int64(69971)

In [10]:
dist = word_counts ** 0.75
dist = dist / dist.sum()

In [11]:
def convert_sentences_to_indices(sentences, word2idx):
    indexed_sentences = []
    for sent in sentences:
        indexed_sent = [word2idx[w] for w in sent]
        if len(indexed_sent) > 1:   # avoid very short sentences
            indexed_sentences.append(indexed_sent)
    return indexed_sentences

indexed_sentences = convert_sentences_to_indices(filtered_sentances, word2idx)

In [12]:
indexed_sentences[0][:10]

[22504, 25095, 1013, 8512, 18384, 19722, 7280, 6467, 7742, 20589]

In [13]:
import torch
from torch.utils.data import Dataset
import random

In [14]:
class SkipGramDataset(Dataset):
    def __init__(self, sentences, window_size):
        self.sentences = sentences
        self.window_size = window_size

        # store all (sentence_id, word_position)
        self.positions = []
        for s_id, sent in enumerate(sentences):
            for w_id in range(len(sent)):
                self.positions.append((s_id, w_id))

    def __len__(self):
        return len(self.positions)

    def __getitem__(self, idx):
        # get sentence id and word position
        sent_id, word_pos = self.positions[idx]
        sentence = self.sentences[sent_id]
    
        # center word
        center_word = sentence[word_pos]
    
        # context window boundaries
        start = max(0, word_pos - self.window_size)
        end = min(len(sentence), word_pos + self.window_size + 1)
    
        # collect context candidates
        context_candidates = [
            sentence[i] for i in range(start, end) if i != word_pos
        ]
    
        # randomly choose ONE context word
        context_word = random.choice(context_candidates)
    
        return (
            torch.tensor(center_word, dtype=torch.long),
            torch.tensor(context_word, dtype=torch.long),
        )

In [15]:
dataset = SkipGramDataset(indexed_sentences, window_size=2)

center, context = dataset[0]
center, context

(tensor(22504), tensor(1013))

In [16]:
type(center), type(context)

(torch.Tensor, torch.Tensor)

In [17]:
center.item() != context.item()

True

In [18]:
import torch
import torch.nn as nn

In [19]:
class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()

        self.in_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.out_embeddings = nn.Embedding(vocab_size, embedding_dim)

    def forward(self, center_words, pos_context_words, neg_context_words):
        """
        center_words:      (B,)
        pos_context_words: (B,)
        neg_context_words: (B, K)
        """

        # embeddings
        center_emb = self.in_embeddings(center_words)          # (B, D)
        pos_emb = self.out_embeddings(pos_context_words)       # (B, D)
        neg_emb = self.out_embeddings(neg_context_words)       # (B, K, D)

        # positive score
        pos_score = torch.sum(center_emb * pos_emb, dim=1)     # (B,)

        # negative score
        neg_score = torch.bmm(
            neg_emb, center_emb.unsqueeze(2)
        ).squeeze(2)                                           # (B, K)

        return pos_score, neg_score

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [23]:
from torch.utils.data import DataLoader

batch_size = 512  # safe for Kaggle GPU

train_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

In [24]:
neg_sampling_dist = torch.tensor(dist, dtype=torch.float)

In [25]:
def sample_negative(batch_size, k, neg_sampling_dist, device):
    """
    batch_size: int
    k: number of negative samples
    neg_sampling_dist: torch tensor of shape (V,)
    device: cuda / cpu
    """
    neg_samples = torch.multinomial(
        neg_sampling_dist,
        num_samples=batch_size * k,
        replacement=True
    )

    neg_samples = neg_samples.view(batch_size, k)
    return neg_samples.to(device)

In [ ]:
embedding_dim = 200
num_epochs = 10
learning_rate = 0.001
k = 5                # negatives per positive

model = SkipGramModel(len(vocab), embedding_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = torch.nn.BCEWithLogitsLoss()

In [ ]:
model.train()

for epoch in range(num_epochs):
    total_loss = 0.0

    for center_batch, context_batch in train_loader:
        # move data to GPU
        center_batch = center_batch.to(device)
        context_batch = context_batch.to(device)

        batch_size = center_batch.size(0)

        # sample negatives
        neg_batch = sample_negative(
            batch_size=batch_size,
            k=k,
            neg_sampling_dist=neg_sampling_dist,
            device=device
        )

        # forward pass
        pos_score, neg_score = model(center_batch, context_batch, neg_batch)

        # labels
        pos_labels = torch.ones_like(pos_score).to(device)
        neg_labels = torch.zeros_like(neg_score).to(device)

        # loss
        loss_pos = criterion(pos_score, pos_labels)
        loss_neg = criterion(neg_score, neg_labels)
        loss = loss_pos + loss_neg

        # backward + update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

Epoch 1/50 | Loss: 8.8380
Epoch 2/50 | Loss: 6.4666
Epoch 3/50 | Loss: 5.2190
Epoch 4/50 | Loss: 4.4150
Epoch 5/50 | Loss: 3.8351
Epoch 6/50 | Loss: 3.4010
Epoch 7/50 | Loss: 3.0614
Epoch 8/50 | Loss: 2.7836
Epoch 9/50 | Loss: 2.5488
Epoch 10/50 | Loss: 2.3434
Epoch 11/50 | Loss: 2.1667
Epoch 12/50 | Loss: 2.0205
Epoch 13/50 | Loss: 1.8919
Epoch 14/50 | Loss: 1.7824
Epoch 15/50 | Loss: 1.6860
Epoch 16/50 | Loss: 1.6060
Epoch 17/50 | Loss: 1.5317
Epoch 18/50 | Loss: 1.4695
Epoch 19/50 | Loss: 1.4137
Epoch 20/50 | Loss: 1.3673
Epoch 21/50 | Loss: 1.3226
Epoch 22/50 | Loss: 1.2855
Epoch 23/50 | Loss: 1.2481
Epoch 24/50 | Loss: 1.2194
Epoch 25/50 | Loss: 1.1904
Epoch 26/50 | Loss: 1.1643
Epoch 27/50 | Loss: 1.1428
Epoch 28/50 | Loss: 1.1212
Epoch 29/50 | Loss: 1.1027
Epoch 30/50 | Loss: 1.0849
Epoch 31/50 | Loss: 1.0669
Epoch 32/50 | Loss: 1.0530
Epoch 33/50 | Loss: 1.0396


In [ ]:
word_vectors = model.in_embeddings.weight.detach().cpu().numpy()

In [ ]:
# L2 normalize (VERY IMPORTANT for cosine similarity)
norms = np.linalg.norm(word_vectors, axis=1, keepdims=True)
word_vectors = word_vectors / norms

In [ ]:
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2)

In [ ]:
i = word2idx["the"]
j = word2idx["of"]

cosine_similarity(word_vectors[i], word_vectors[j])

In [ ]:
def nearest_neighbors(word, word_vectors, word2idx, idx2word, top_k=10):
    if word not in word2idx:
        print(f"'{word}' not in vocabulary")
        return

    word_idx = word2idx[word]
    query_vec = word_vectors[word_idx]

    # cosine similarity with all words
    similarities = np.dot(word_vectors, query_vec)

    # sort by similarity (descending)
    best_indices = np.argsort(similarities)[::-1]

    print(f"\nNearest neighbors for '{word}':\n")

    count = 0
    for idx in best_indices:
        if idx == word_idx:
            continue
        print(f"{idx2word[idx]:15s}  sim={similarities[idx]:.4f}")
        count += 1
        if count == top_k:
            break

In [ ]:
nearest_neighbors("king", word_vectors, word2idx, idx2word)
nearest_neighbors("dog", word_vectors, word2idx, idx2word)
nearest_neighbors("woman", word_vectors, word2idx, idx2word)

In [ ]:
def word_similarity(w1, w2, word_vectors, word2idx):
    if w1 not in word2idx or w2 not in word2idx:
        return None
    return cosine_similarity(
        word_vectors[word2idx[w1]],
        word_vectors[word2idx[w2]]
    )

In [ ]:
word_similarity("man", "woman", word_vectors, word2idx), word_similarity("man", "dog", word_vectors, word2idx)

In [ ]:
def analogy(a, b, c, word_vectors, word2idx, idx2word, top_k=5):
    for w in (a, b, c):
        if w not in word2idx:
            print(f"'{w}' not in vocabulary")
            return

    a_idx = word2idx[a]
    b_idx = word2idx[b]
    c_idx = word2idx[c]

    # compute target vector
    target_vec = word_vectors[b_idx] - word_vectors[a_idx] + word_vectors[c_idx]

    # normalize target
    target_vec = target_vec / np.linalg.norm(target_vec)

    # cosine similarity with all words
    similarities = np.dot(word_vectors, target_vec)

    # sort by similarity
    best_indices = np.argsort(similarities)[::-1]

    print(f"\nAnalogy: {a} : {b} :: {c} : ?\n")

    count = 0
    for idx in best_indices:
        word = idx2word[idx]
        if word in (a, b, c):
            continue
        print(f"{word:15s} sim={similarities[idx]:.4f}")
        count += 1
        if count == top_k:
            break

In [ ]:
analogy("man", "king", "woman", word_vectors, word2idx, idx2word)
analogy("woman", "queen", "man", word_vectors, word2idx, idx2word)
analogy("paris", "france", "london", word_vectors, word2idx, idx2word)
analogy("good", "better", "bad", word_vectors, word2idx, idx2word)